<a href="https://colab.research.google.com/github/SinnottKayleigh/B2B-Sales-Algos/blob/main/CRM_Integration_(test).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Flexible CRM Integration:
Abstract base class for CRM connectors
Specific implementations for different CRMs
Easy addition of new CRM sources
Data Standardization:
Consistent schema across sources
Field mapping and transformation
Data validation and cleaning
Deduplication:
Fuzzy matching for similar records
Business rule-based deduplication
Source priority handling
Data Enrichment:
Combined view of all sources
Additional computed fields
Data quality improvements

In [1]:
from typing import Dict, List, Optional
import pandas as pd
from datetime import datetime
import numpy as np
from sqlalchemy import create_engine
import requests
from abc import ABC, abstractmethod

class CRMConnector(ABC):
    """
    Abstract base class for CRM connections
    """
    @abstractmethod
    def connect(self):
        pass

    @abstractmethod
    def fetch_data(self):
        pass

    @abstractmethod
    def transform_data(self, data):
        pass

class SalesforceConnector(CRMConnector):
    def __init__(self, credentials: Dict):
        self.credentials = credentials
        self.client = None

    def connect(self):
        """
        Connect to Salesforce API
        """
        try:
            from simple_salesforce import Salesforce
            self.client = Salesforce(
                username=self.credentials['username'],
                password=self.credentials['password'],
                security_token=self.credentials['security_token']
            )
            return True
        except Exception as e:
            print(f"Salesforce connection error: {e}")
            return False

    def fetch_data(self) -> Dict:
        """
        Fetch relevant data from Salesforce
        """
        data = {
            'opportunities': self.client.query("SELECT Id, Name, StageName, Amount, CloseDate FROM Opportunity"),
            'accounts': self.client.query("SELECT Id, Name, Industry, Type FROM Account"),
            'contacts': self.client.query("SELECT Id, Name, Email, Title FROM Contact")
        }
        return data

    def transform_data(self, data: Dict) -> Dict:
        """
        Transform Salesforce data to Prospectoro format
        """
        return {
            'opportunities': pd.DataFrame(data['opportunities']['records']),
            'accounts': pd.DataFrame(data['accounts']['records']),
            'contacts': pd.DataFrame(data['contacts']['records'])
        }

class HubspotConnector(CRMConnector):
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.hubapi.com"

    def connect(self):
        self.headers = {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json'
        }
        return True

    def fetch_data(self) -> Dict:
        """
        Fetch relevant data from HubSpot
        """
        data = {
            'deals': self._get_deals(),
            'companies': self._get_companies(),
            'contacts': self._get_contacts()
        }
        return data

    def transform_data(self, data: Dict) -> Dict:
        """
        Transform HubSpot data to Prospectoro format
        """
        return {
            'opportunities': self._transform_deals(data['deals']),
            'accounts': self._transform_companies(data['companies']),
            'contacts': self._transform_contacts(data['contacts'])
        }

class DataIntegrationPipeline:
    def __init__(self):
        self.connectors = {}
        self.standardized_schema = {
            'opportunities': {
                'required_fields': ['id', 'name', 'stage', 'value', 'close_date'],
                'optional_fields': ['probability', 'source', 'last_activity']
            },
            'accounts': {
                'required_fields': ['id', 'name', 'industry'],
                'optional_fields': ['size', 'revenue', 'location']
            },
            'contacts': {
                'required_fields': ['id', 'name', 'email', 'role'],
                'optional_fields': ['phone', 'engagement_score', 'last_contact']
            }
        }

    def add_connector(self, name: str, connector: CRMConnector):
        """
        Add a CRM connector to the pipeline
        """
        self.connectors[name] = connector

    def process_all_sources(self) -> Dict:
        """
        Process data from all CRM sources
        """
        combined_data = {
            'opportunities': [],
            'accounts': [],
            'contacts': []
        }

        for source, connector in self.connectors.items():
            try:
                # Fetch and transform data
                raw_data = connector.fetch_data()
                transformed_data = connector.transform_data(raw_data)

                # Add source tracking
                for key in transformed_data:
                    transformed_data[key]['data_source'] = source
                    combined_data[key].append(transformed_data[key])

            except Exception as e:
                print(f"Error processing {source}: {e}")

        # Combine and deduplicate data
        return self.combine_and_deduplicate(combined_data)

    def combine_and_deduplicate(self, combined_data: Dict) -> Dict:
        """
        Combine and deduplicate data from different sources
        """
        result = {}
        for key in combined_data:
            # Combine data from all sources
            df = pd.concat(combined_data[key], ignore_index=True)

            # Deduplicate based on business rules
            df = self.deduplicate_records(df, key)

            # Standardize schema
            df = self.standardize_schema(df, key)

            result[key] = df

        return result

    def deduplicate_records(self, df: pd.DataFrame, entity_type: str) -> pd.DataFrame:
        """
        Deduplicate records using fuzzy matching
        """
        from fuzzywuzzy import fuzz

        def find_duplicates(group):
            duplicates = []
            for i in range(len(group)):
                for j in range(i + 1, len(group)):
                    similarity = fuzz.ratio(group.iloc[i]['name'], group.iloc[j]['name'])
                    if similarity > 90:  # Threshold for considering records as duplicates
                        duplicates.append(j)
            return list(set(duplicates))

        # Group by relevant fields and find duplicates
        if entity_type == 'contacts':
            groups = df.groupby('email')
        else:
            groups = df.groupby('name')

        duplicates = []
        for name, group in groups:
            duplicates.extend(find_duplicates(group))

        # Keep the most recent/complete record
        return df.drop(duplicates)

    def standardize_schema(self, df: pd.DataFrame, entity_type: str) -> pd.DataFrame:
        """
        Standardize dataframe schema according to Prospectoro requirements
        """
        required_fields = self.standardized_schema[entity_type]['required_fields']
        optional_fields = self.standardized_schema[entity_type]['optional_fields']

        # Ensure required fields exist
        for field in required_fields:
            if field not in df.columns:
                df[field] = None

        # Add optional fields if they don't exist
        for field in optional_fields:
            if field not in df.columns:
                df[field] = None

        # Select only relevant fields
        return df[required_fields + optional_fields]

class ProspectoroDataManager:
    def __init__(self, db_connection_string: str):
        self.engine = create_engine(db_connection_string)
        self.integration_pipeline = DataIntegrationPipeline()

    def setup_crm_connections(self, crm_configs: Dict):
        """
        Setup connections to various CRM systems
        """
        for crm_name, config in crm_configs.items():
            if crm_name == 'salesforce':
                connector = SalesforceConnector(config)
            elif crm_name == 'hubspot':
                connector = HubspotConnector(config['api_key'])
            # Add more CRM connectors as needed

            self.integration_pipeline.add_connector(crm_name, connector)

    def update_data(self):
        """
        Update Prospectoro data from all CRM sources
        """
        # Process all CRM sources
        combined_data = self.integration_pipeline.process_all_sources()

        # Update database
        for entity_type, df in combined_data.items():
            df.to_sql(
                entity_type,
                self.engine,
                if_exists='replace',
                index=False
            )

    def get_enriched_data(self, entity_type: str, filters: Dict = None) -> pd.DataFrame:
        """
        Get enriched data from Prospectoro database
        """
        query = f"SELECT * FROM {entity_type}"
        if filters:
            conditions = " AND ".join([f"{k} = '{v}'" for k, v in filters.items()])
            query += f" WHERE {conditions}"

        return pd.read_sql(query, self.engine)

# Usage example:
def setup_prospectoro_data_integration():
    # Configuration
    crm_configs = {
        'salesforce': {
            'username': 'your_username',
            'password': 'your_password',
            'security_token': 'your_security_token'
        },
        'hubspot': {
            'api_key': 'your_api_key'
        }
    }

    # Initialize data manager
    data_manager = ProspectoroDataManager('your_db_connection_string')

    # Setup CRM connections
    data_manager.setup_crm_connections(crm_configs)

    # Update data
    data_manager.update_data()

    # Get enriched data
    opportunities = data_manager.get_enriched_data(
        'opportunities',
        filters={'stage': 'Negotiation'}
    )

    return data_manager

Setup Database:

In [2]:
# PostgreSQL setup
db_config = {
    'host': 'your_host',
    'database': 'prospectoro',
    'user': 'your_user',
    'password': 'your_password'
}

connection_string = f"postgresql://{db_config['user']}:{db_config['password']}@{db_config['host']}/{db_config['database']}"

Configure CRM Connections:

In [3]:
crm_configs = {
    'salesforce': {
        'username': 'sf_username',
        'password': 'sf_password',
        'security_token': 'sf_token'
    },
    'hubspot': {
        'api_key': 'hubspot_api_key'
    }
}

Schedule Updates:


In [10]:
from apscheduler.schedulers.background import BackgroundScheduler

scheduler = BackgroundScheduler()
scheduler.add_job(
    data_manager.update_data,
    'interval',
    hours=6
)
scheduler.start()

NameError: name 'data_manager' is not defined

In [5]:
pip install apscheduler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 2.6 MB/s eta 0:00:00


Extracting the data to aid in algorithm learning

In [6]:
class ProspectoroAIDataIntegration:
    def __init__(self):
        self.data_processors = {
            'crm': CRMDataProcessor(),
            'enrichment': DataEnrichmentProcessor(),
            'ml': MLDataProcessor()
        }
        self.ml_models = {
            'lead_scoring': LeadScoringModel(),
            'pattern_recognition': PatternRecognitionModel(),
            'recommendation': RecommendationModel()
        }

class CRMDataProcessor:
    """
    Process and analyze historical CRM data
    """
    def __init__(self):
        self.connectors = {
            'salesforce': SalesforceConnector(),
            'hubspot': HubspotConnector(),
            'pipedrive': PipedriveConnector(),
            'zoho': ZohoConnector()
        }

    def extract_success_patterns(self, historical_data: pd.DataFrame) -> Dict:
        """
        Extract successful sales patterns from historical data
        """
        success_patterns = {
            'ideal_customer_profile': self.analyze_successful_customers(historical_data),
            'effective_sequences': self.analyze_successful_sequences(historical_data),
            'timing_patterns': self.analyze_timing_patterns(historical_data),
            'stakeholder_patterns': self.analyze_stakeholder_patterns(historical_data)
        }
        return success_patterns

    def analyze_successful_customers(self, data: pd.DataFrame) -> Dict:
        """
        Analyze characteristics of successful customers
        """
        won_deals = data[data['status'] == 'won']
        return {
            'industry_distribution': won_deals.groupby('industry').size().to_dict(),
            'company_size_distribution': won_deals.groupby('company_size').size().to_dict(),
            'average_deal_value': won_deals['deal_value'].mean(),
            'common_technologies': self.extract_common_technologies(won_deals),
            'decision_maker_roles': self.analyze_decision_makers(won_deals)
        }

class DataEnrichmentProcessor:
    """
    Enrich CRM data with additional insights
    """
    def __init__(self):
        self.enrichment_sources = {
            'company': CompanyDataEnricher(),
            'contact': ContactDataEnricher(),
            'technology': TechnologyStackEnricher(),
            'intent': IntentDataEnricher()
        }

    def enrich_lead_data(self, lead_data: Dict) -> Dict:
        """
        Enrich lead data with additional information
        """
        enriched_data = lead_data.copy()

        # Enrich company information
        company_info = self.enrichment_sources['company'].enrich(
            lead_data.get('company_name'),
            lead_data.get('domain')
        )

        # Enrich contact information
        contact_info = self.enrichment_sources['contact'].enrich(
            lead_data.get('contact_email'),
            lead_data.get('contact_name')
        )

        # Get technology stack
        tech_stack = self.enrichment_sources['technology'].enrich(
            lead_data.get('domain')
        )

        # Get intent signals
        intent_data = self.enrichment_sources['intent'].enrich(
            lead_data.get('domain'),
            lead_data.get('company_name')
        )

        enriched_data.update({
            'company_info': company_info,
            'contact_info': contact_info,
            'technology_stack': tech_stack,
            'intent_signals': intent_data
        })

        return enriched_data

class MLDataProcessor:
    """
    Process data for machine learning models
    """
    def __init__(self):
        self.feature_extractors = {
            'company': CompanyFeatureExtractor(),
            'contact': ContactFeatureExtractor(),
            'engagement': EngagementFeatureExtractor(),
            'sequence': SequenceFeatureExtractor()
        }

    def prepare_training_data(self, historical_data: pd.DataFrame) -> Tuple[np.array, np.array]:
        """
        Prepare historical data for ML model training
        """
        features = []
        labels = []

        for _, row in historical_data.iterrows():
            # Extract features
            company_features = self.feature_extractors['company'].extract(row)
            contact_features = self.feature_extractors['contact'].extract(row)
            engagement_features = self.feature_extractors['engagement'].extract(row)
            sequence_features = self.feature_extractors['sequence'].extract(row)

            # Combine features
            combined_features = np.concatenate([
                company_features,
                contact_features,
                engagement_features,
                sequence_features
            ])

            features.append(combined_features)
            labels.append(row['converted'])

        return np.array(features), np.array(labels)

class LeadScoringModel:
    """
    AI model for lead scoring
    """
    def __init__(self):
        self.model = XGBClassifier(
            max_depth=6,
            learning_rate=0.1,
            n_estimators=100
        )
        self.feature_importance = None

    def train(self, X: np.array, y: np.array):
        """
        Train the lead scoring model
        """
        self.model.fit(X, y)
        self.feature_importance = self.model.feature_importances_

    def predict_score(self, lead_features: np.array) -> float:
        """
        Predict lead score
        """
        return self.model.predict_proba(lead_features.reshape(1, -1))[0][1]

class PatternRecognitionModel:
    """
    Recognize successful patterns in sales data
    """
    def __init__(self):
        self.sequence_model = LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=2
        )
        self.pattern_classifier = RandomForestClassifier(
            n_estimators=100,
            max_depth=None
        )

    def identify_patterns(self, sequence_data: List[Dict]) -> Dict:
        """
        Identify successful patterns in sales sequences
        """
        return {
            'timing_patterns': self.analyze_timing(sequence_data),
            'channel_patterns': self.analyze_channels(sequence_data),
            'content_patterns': self.analyze_content(sequence_data),
            'stakeholder_patterns': self.analyze_stakeholders(sequence_data)
        }

class RecommendationModel:
    """
    Generate recommendations for lead generation and engagement
    """
    def __init__(self):
        self.content_recommender = ContentBasedRecommender()
        self.sequence_recommender = SequenceRecommender()
        self.timing_recommender = TimingRecommender()

    def generate_recommendations(self, lead_data: Dict, historical_patterns: Dict) -> Dict:
        """
        Generate personalized recommendations
        """
        return {
            'next_best_action': self.sequence_recommender.recommend_next_action(lead_data),
            'content_recommendations': self.content_recommender.recommend_content(lead_data),
            'timing_recommendations': self.timing_recommender.recommend_timing(lead_data),
            'personalization_suggestions': self.generate_personalization_suggestions(lead_data)
        }

# Usage Example
def setup_prospectoro_ai_system():
    # Initialize the AI system
    ai_system = ProspectoroAIDataIntegration()

    # Process historical CRM data
    historical_data = ai_system.data_processors['crm'].process_historical_data()

    # Enrich data
    enriched_data = ai_system.data_processors['enrichment'].enrich_data(historical_data)

    # Prepare for ML
    X, y = ai_system.data_processors['ml'].prepare_training_data(enriched_data)

    # Train models
    ai_system.ml_models['lead_scoring'].train(X, y)

    # Generate insights
    success_patterns = ai_system.data_processors['crm'].extract_success_patterns(historical_data)

    return ai_system, success_patterns

NameError: name 'Tuple' is not defined

Add API endpoints for data integration:

In [7]:
@app.route('/api/v1/integrate/crm', methods=['POST'])
def integrate_crm():
    crm_type = request.json['crm_type']
    credentials = request.json['credentials']
    return ai_system.integrate_crm(crm_type, credentials)

NameError: name 'app' is not defined

Add background tasks for processing:

In [8]:
 @celery.task
def process_crm_data(crm_type, data):
    return ai_system.process_crm_data(crm_type, data)

NameError: name 'celery' is not defined

Add monitoring and feedback loops:




In [9]:
@app.route('/api/v1/feedback', methods=['POST'])
def record_feedback():
    feedback_data = request.json
    return ai_system.update_models(feedback_data)

NameError: name 'app' is not defined

APIs Required

In [11]:
from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel
from typing import Dict, List, Optional
import asyncio

app = FastAPI()

# API Models/Schemas
class CRMCredentials(BaseModel):
    crm_type: str
    credentials: Dict

class CompanyData(BaseModel):
    name: str
    domain: str
    industry: Optional[str]
    size: Optional[str]

class ContactData(BaseModel):
    email: str
    name: str
    title: Optional[str]
    company: Optional[str]

class EnrichmentResponse(BaseModel):
    company_info: Dict
    contact_info: Dict
    technology_stack: List[str]
    intent_signals: Dict

# 1. CRM Integration APIs
@app.post("/api/v1/crm/connect")
async def connect_crm(credentials: CRMCredentials):
    """
    Connect to a CRM system
    """
    try:
        # Initialize appropriate CRM connector
        if credentials.crm_type == "salesforce":
            connector = SalesforceConnector(credentials.credentials)
        elif credentials.crm_type == "hubspot":
            connector = HubspotConnector(credentials.credentials)
        else:
            raise HTTPException(status_code=400, detail="Unsupported CRM type")

        # Test connection
        connection_status = await connector.test_connection()

        return {"status": "success", "connection_id": connection_status.id}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/v1/crm/sync")
async def sync_crm_data(connection_id: str):
    """
    Sync data from connected CRM
    """
    try:
        sync_task = await start_sync_task(connection_id)
        return {"status": "syncing", "task_id": sync_task.id}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 2. Data Enrichment APIs
@app.post("/api/v1/enrich/company")
async def enrich_company_data(company: CompanyData):
    """
    Enrich company data using various data sources
    """
    try:
        # Company enrichment services
        clearbit_data = await clearbit_client.enrich_company(company.domain)
        zoominfo_data = await zoominfo_client.get_company_info(company.name)

        return {
            "clearbit": clearbit_data,
            "zoominfo": zoominfo_data
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/v1/enrich/contact")
async def enrich_contact_data(contact: ContactData):
    """
    Enrich contact data
    """
    try:
        # Contact enrichment services
        apollo_data = await apollo_client.enrich_contact(contact.email)
        hunter_data = await hunter_client.verify_email(contact.email)

        return {
            "apollo": apollo_data,
            "hunter": hunter_data
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 3. Technology Detection API
@app.get("/api/v1/detect/technology/{domain}")
async def detect_technology(domain: str):
    """
    Detect technology stack for a domain
    """
    try:
        builtwith_data = await builtwith_client.lookup(domain)
        return builtwith_data
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 4. Intent Signal APIs
@app.get("/api/v1/intent/{company_domain}")
async def get_intent_signals(company_domain: str):
    """
    Get buying intent signals
    """
    try:
        bombora_data = await bombora_client.get_intent_data(company_domain)
        g2_data = await g2_client.get_company_research(company_domain)

        return {
            "bombora": bombora_data,
            "g2": g2_data
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 5. AI Model APIs
@app.post("/api/v1/ai/score-lead")
async def score_lead(lead_data: Dict):
    """
    Score a lead using AI model
    """
    try:
        score = await ai_service.score_lead(lead_data)
        return {"score": score}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/api/v1/ai/recommend-actions")
async def recommend_actions(lead_data: Dict):
    """
    Get AI recommendations for next actions
    """
    try:
        recommendations = await ai_service.get_recommendations(lead_data)
        return recommendations
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 6. Integration Status APIs
@app.get("/api/v1/integration/status/{task_id}")
async def get_integration_status(task_id: str):
    """
    Get status of integration task
    """
    try:
        status = await check_task_status(task_id)
        return status
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# API Clients for Third-party Services
class ClearbitClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://company.clearbit.com/v2"

    async def enrich_company(self, domain: str) -> Dict:
        # Implementation for Clearbit API calls
        pass

class ApolloClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.apollo.io/v1"

    async def enrich_contact(self, email: str) -> Dict:
        # Implementation for Apollo API calls
        pass

# Configuration for Third-party APIs
api_config = {
    "clearbit": {
        "api_key": "your_clearbit_api_key",
        "base_url": "https://company.clearbit.com/v2"
    },
    "apollo": {
        "api_key": "your_apollo_api_key",
        "base_url": "https://api.apollo.io/v1"
    },
    "zoominfo": {
        "api_key": "your_zoominfo_api_key",
        "base_url": "https://api.zoominfo.com/v1"
    },
    "hunter": {
        "api_key": "your_hunter_api_key",
        "base_url": "https://api.hunter.io/v2"
    },
    "builtwith": {
        "api_key": "your_builtwith_api_key",
        "base_url": "https://api.builtwith.com/v1"
    },
    "bombora": {
        "api_key": "your_bombora_api_key",
        "base_url": "https://api.bombora.com/v1"
    }
}

# Initialize API clients
clearbit_client = ClearbitClient(api_config["clearbit"]["api_key"])
apollo_client = ApolloClient(api_config["apollo"]["api_key"])
# ... initialize other clients

ModuleNotFoundError: No module named 'fastapi'

In [12]:
pip install fastapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.1 MB/s eta 0:00:00
